In [2]:
from sqlalchemy import create_engine
import pandas as pd 
import getpass 
password = getpass.getpass()

In [8]:
engine = create_engine("mysql+pymysql://root:" + password + "@localhost:3306/sakila")

def rentals_month(engine,month,year):
    query = '''
    SELECT
        rental_id,
        rental_date,
        customer_id
    FROM rental
    WHERE MONTH(rental_date) =%s
    AND YEAR(rental_date) =%s
    '''
    df = pd.read_sql(query,engine, params =(month, year))
    return df

In [11]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"
    
    result = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )
    
    return result


In [12]:
def compare_rentals(df1, df2):
    combined = pd.merge(df1, df2, on="customer_id", how="outer")
    
    combined = combined.fillna(0)
    
    rental_cols = combined.columns.drop("customer_id")
    
    combined["difference"] = combined[rental_cols[1]] - combined[rental_cols[0]]
    
    return combined

In [15]:
may_df = rentals_month(engine, 5, 2005)
june_df = rentals_month(engine, 6, 2005)

may_counts = rental_count_month(may_df, 5, 2005)
june_counts = rental_count_month(june_df, 6, 2005)

comparison = compare_rentals(may_counts, june_counts)

print(comparison.head())

   customer_id  rentals_05_2005  rentals_06_2005  difference
0            1              2.0              7.0         5.0
1            2              1.0              1.0         0.0
2            3              2.0              4.0         2.0
3            4              0.0              6.0         6.0
4            5              3.0              5.0         2.0
